In [1]:
import pandas as pd

file = "mobile_shoppee_curation_project.xlsx"

sheet_names = pd.ExcelFile(file).sheet_names
print(sheet_names)

['Customers', 'MobileModels', 'ShopLocations', 'Sales', 'Stocks', 'Offers', 'DataDictionary']


In [2]:
import pandas as pd
sales = pd.read_csv(r"C:\Users\katka\OneDrive\ドキュメント\EDA_Project/Sales.csv")
models = pd.read_csv(r"C:\Users\katka\OneDrive\ドキュメント\EDA_Project/MobileModels.csv")

In [3]:
#Apriori Algorithm step 1

import pandas as pd
from itertools import combinations
from collections import defaultdict

# Load your sales dataset
sales = pd.read_csv("Sales.csv")

def make_basket(row):
    basket = [
        f"brand_{row['brand']}",
        f"ram_{int(row['ram_gb'])}",
        f"storage_{int(row['storage_gb'])}"
    ]

    price = row['base_price']
    if price < 10000: pr = "low"
    elif price < 20000: pr = "mid"
    elif price < 30000: pr = "high"
    else: pr = "premium"

    basket.append(f"price_{pr}")
    return basket

# Convert all rows into transactional format
transactions = sales.apply(make_basket, axis=1).tolist()


In [4]:
def apriori(transactions, min_support=0.05):
    total = len(transactions)
    min_count = total * min_support

    # Count single items
    item_counts = defaultdict(int)
    for trx in transactions:
        for item in trx:
            item_counts[item] += 1

    freq_items = {frozenset([i]): c for i,c in item_counts.items() if c >= min_count}            #keep items above support

    all_freq = freq_items.copy()
    current_freq = freq_items
    k = 2

    while current_freq:
        candidates = []
        keys = list(current_freq.keys())

        # self join to make k-itemsets
        for i in range(len(keys)):
            for j in range(i+1, len(keys)):
                union = keys[i] | keys[j]
                if len(union) == k:
                    candidates.append(union)

        # count candidate frequency
        candidate_counts = defaultdict(int)
        for trx in transactions:
            tset = set(trx)
            for c in candidates:
                if c.issubset(tset):
                    candidate_counts[c] += 1

        current_freq = {itemset:cnt for itemset,cnt in candidate_counts.items() if cnt >= min_count}

        all_freq.update(current_freq)
        k += 1

    # Convert to DataFrame
    freq_df = pd.DataFrame([
        {"itemset": item, "support": cnt/total}
        for item, cnt in all_freq.items()
    ])

    return freq_df


In [5]:
def generate_rules(freq_df, min_conf=0.4):
    rules = []

    for _, row in freq_df.iterrows():
        itemset = row["itemset"]
        support_itemset = row["support"]
        items = list(itemset)

        if len(items) < 2:
            continue

        for i in range(1, len(items)):
            for antecedent in combinations(items, i):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent

                sup_a = freq_df[freq_df["itemset"] == antecedent]["support"]
                if len(sup_a) == 0:
                    continue

                conf = support_itemset / sup_a.values[0]

                if conf >= min_conf:
                    rules.append({
                        "antecedent": antecedent,
                        "consequent": consequent,
                        "support": support_itemset,
                        "confidence": conf
                    })

    return pd.DataFrame(rules)


In [6]:
freq = apriori(transactions)
rules = generate_rules(freq)

ram_storage_rules = rules[
    rules["antecedent"].astype(str).str.contains("ram_") &
    rules["consequent"].astype(str).str.contains("storage_")
]

print("🔹 RAM + STORAGE ASSOCIATION RULES")
print(ram_storage_rules)


🔹 RAM + STORAGE ASSOCIATION RULES
                  antecedent                 consequent   support  confidence
12                   (ram_8)               (storage_64)  0.074575    0.404091
16       (price_high, ram_4)               (storage_64)  0.119700    0.893117
21       (price_high, ram_4)              (storage_256)  0.103950    0.775602
23                   (ram_6)  (price_high, storage_128)  0.121200    0.504632
25       (price_high, ram_6)              (storage_128)  0.121200    0.989388
28   (ram_12, price_premium)               (storage_64)  0.060900    0.762919
31                  (ram_12)   (price_high, storage_64)  0.122925    0.467573
33      (price_high, ram_12)               (storage_64)  0.122925    0.955314
40        (ram_4, price_mid)              (storage_128)  0.089175    1.030032
44                   (ram_6)  (price_high, storage_512)  0.155625    0.647965
46       (price_high, ram_6)              (storage_512)  0.155625    1.270408
48                   (ram_8)  

In [7]:
brand_price_rules = rules[
    rules["antecedent"].astype(str).str.contains("brand_") &
    rules["consequent"].astype(str).str.contains("price_")
]

print("🔹 BRAND → PRICE RANGE RULES")
print(brand_price_rules)


🔹 BRAND → PRICE RANGE RULES
                      antecedent                 consequent   support  \
0                 (brand_realme)               (price_high)  0.063975   
3                (brand_Samsung)               (price_high)  0.063350   
5                  (brand_Apple)               (price_high)  0.083400   
8                (brand_Samsung)                (price_mid)  0.054625   
10               (brand_OnePlus)               (price_high)  0.069550   
14              (brand_Motorola)               (price_high)  0.052625   
36               (brand_OnePlus)       (price_high, ram_12)  0.073275   
39       (ram_12, brand_OnePlus)               (price_high)  0.073275   
74                  (brand_OPPO)       (price_high, ram_12)  0.074775   
77          (brand_OPPO, ram_12)               (price_high)  0.074775   
81               (brand_OnePlus)  (price_high, storage_128)  0.089700   
84  (brand_OnePlus, storage_128)               (price_high)  0.089700   

    confidence  
0    

In [8]:
price_feature_rules = rules[
    rules["antecedent"].astype(str).str.contains("price_") |
    rules["consequent"].astype(str).str.contains("price_")
]

print("🔹 PRICE + FEATURE COMBINATIONS")
print(price_feature_rules)


🔹 PRICE + FEATURE COMBINATIONS
                       antecedent       consequent   support  confidence
0                  (brand_realme)     (price_high)  0.063975    0.419371
1                         (ram_4)     (price_high)  0.134025    0.429052
2                    (storage_64)     (price_high)  0.130500    0.472783
3                 (brand_Samsung)     (price_high)  0.063350    0.493861
4                   (storage_256)     (price_high)  0.098725    0.512391
..                            ...              ...       ...         ...
98    (storage_64, price_premium)          (ram_4)  0.064350    0.971687
99            (ram_4, storage_64)  (price_premium)  0.064350    0.780000
100       (ram_12, price_premium)    (storage_128)  0.058725    0.735672
101         (ram_12, storage_128)  (price_premium)  0.058725    0.910112
102  (price_premium, storage_128)         (ram_12)  0.058725    0.588574

[102 rows x 4 columns]


In [9]:
top_features = freq.sort_values("support", ascending=False).head(20)

print("🔹 TOP 20 MOST FREQUENT FEATURE COMBOS")
print(top_features)


🔹 TOP 20 MOST FREQUENT FEATURE COMBOS
                             itemset   support
3                       (price_high)  0.463200
1                            (ram_4)  0.312375
8                      (storage_128)  0.299200
2                       (storage_64)  0.276025
9                           (ram_12)  0.262900
10                   (price_premium)  0.261575
7                            (ram_6)  0.240175
14                       (price_mid)  0.240025
13                     (storage_512)  0.232100
5                      (storage_256)  0.192675
16                           (ram_8)  0.184550
15                      (brand_OPPO)  0.157300
68  (price_high, storage_512, ram_6)  0.155625
0                     (brand_realme)  0.152550
17                   (brand_OnePlus)  0.137225
21               (price_high, ram_4)  0.134025
22          (price_high, storage_64)  0.130500
44              (price_high, ram_12)  0.128675
4                    (brand_Samsung)  0.128275
11                  (b